[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/skill_suggestion.ipynb)

# Suggest one intent from a large catalog

A catalog of intents does not fit a polite prompt, and the right answer is often "none of these". FRIDA-Decisions first ranks the catalog in one request — option ids are not shown to the model — then a second request asks a yes/no question of each of the top candidates. Code keeps the candidate with the highest yes-probability, and abstains when that probability is below a cutoff.

`fast=True` ranks a pool of 40 intents that includes every labelled one. Set it to `False` (or pass `--full` to the script) to rank all 243 intents. The catalog is downloaded from `main` if it is not already next to the example. The state of a long dialogue can hit the default 384-token cut; `usage.state_truncated` reports that.

The description of each candidate goes into the `instructions` of its yes/no question. Instructions are cut at 96 tokens, suffix included, which these descriptions fit with room to spare; a catalog with long descriptions would need them in `state` instead.

In [ ]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [1]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


Rank, then re-check the top five. The ranking always names a winner, because its probabilities split one unit across the list. The yes/no pass can refuse all of them.

When two yes-probabilities are within 0.02, keep the one ranking put first. int8 ONNX can swap two answers that both print as 0.98.


In [2]:
import json
import urllib.request
from pathlib import Path

DATA = [
    {"text": "Я клиент другого оператора. Хочу стать вашим абонентом и перенести к вам свой текущий номер.", "intent": "port_in_keep_number"},
    {"text": "Потерял телефон в метро, заблокируйте сим-карту, чтобы с неё никто не звонил.", "intent": "sim_lost_block"},
    {"text": "Какой у меня сейчас тариф и что в него входит?", "intent": "tariff_current_info"},
    {"text": "Хочу заменить пластиковую сим-карту на eSIM.", "intent": "esim_activate"},
]
TOP_K = 5
NOUL_MIN = 0.5
TIE = 0.02
FAST_POOL = 40
CATALOG_URL = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "main/examples/data/intent_catalog.json")


def load_catalog():
    for path in (Path("examples/data/intent_catalog.json"), Path("../data/intent_catalog.json")):
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    with urllib.request.urlopen(CATALOG_URL, timeout=120) as resp:
        return json.loads(resp.read().decode("utf-8"))


def pool_for(criteria, fast):
    if not fast:
        return criteria
    gold = [row["intent"] for row in DATA]
    chosen = []
    for key in gold + list(criteria):
        if key not in chosen:
            chosen.append(key)
        if len(chosen) == FAST_POOL:
            break
    picked = {}
    for key in chosen:
        picked[key] = criteria[key]
    return picked


def suggest(judge, text, criteria):
    ranked = judge({
        "state": text,
        "questions": {"best": {
            "type": "ranking",
            "instructions": "Какое намерение лучше всего описывает сообщение клиента?",
            "criteria": criteria,
        }},
    })
    top = ranked["answers"]["best"]["ranking"][:TOP_K]
    checked = judge({
        "state": text,
        "questions": {
            key: {"type": "noul", "instructions": "Клиент хочет именно это: " + criteria[key]}
            for key in top
        },
    })
    best_noul = 0.0
    for key in top:
        best_noul = max(best_noul, checked["answers"][key]["noul"])
    best = None
    for key in top:
        if best_noul - checked["answers"][key]["noul"] <= TIE:
            best = key
            break
    noul = checked["answers"][best]["noul"]
    intent = None
    if noul >= NOUL_MIN:
        intent = best
    top_rows = []
    for key in top:
        top_rows.append((key, checked["answers"][key]["noul"]))
    truncated = ranked["usage"]["state_truncated"] or checked["usage"]["state_truncated"]
    return {"intent": intent, "noul": noul, "top": top_rows,
            "truncated": truncated, "ranked_ms": ranked["usage"]["milliseconds"]}


def run(judge, fast=True):
    catalog = load_catalog()
    criteria = catalog["request"]["questions"]["intent"]["criteria"]
    picked = pool_for(criteria, fast)
    print(f"pool {len(picked)} of {len(criteria)} intents, top-{TOP_K}, noul cutoff {NOUL_MIN}")
    hits = 0
    for row in DATA:
        result = suggest(judge, row["text"], picked)
        if result["intent"] == row["intent"]:
            mark = "ok"
            hits += 1
        else:
            mark = "miss"
        shown = " ".join(f"{key}:{noul:.2f}" for key, noul in result["top"])
        print(f"{mark:4} got {result['intent']!s:28} gold {row['intent']:28} "
              f"noul {result['noul']:.2f}  {result['ranked_ms']:.0f} ms")
        print(f"     top {shown}")
        if result["truncated"]:
            print("     state truncated: default cut is 384 tokens")
    print(f"\n{hits}/{len(DATA)} match. fast={fast}")


In [3]:
run(judge, fast=True)


pool 40 of 243 intents, top-5, noul cutoff 0.5
ok   got port_in_keep_number          gold port_in_keep_number          noul 0.98  3536 ms
     top port_in_keep_number:0.98 tariff_change:0.94 new_sim_new_number:0.98 change_number_keep_tariff:0.96 port_out_to_other_operator:0.86
ok   got sim_lost_block               gold sim_lost_block               noul 1.00  3564 ms
     top sim_lost_block:1.00 number_restore_after_block:0.27 sim_replace:0.24 close_number:0.64 esim_transfer_new_phone:0.09
ok   got tariff_current_info          gold tariff_current_info          noul 0.99  3519 ms
     top tariff_current_info:0.99 tariff_change_cost:0.79 tariff_choose:0.88 tariff_minutes_left:0.87 tariff_change:0.31
ok   got esim_activate                gold esim_activate                noul 0.98  3507 ms
     top esim_activate:0.98 esim_transfer_new_phone:0.95 new_sim_new_number:0.96 sim_for_smartwatch:0.73 sim_replace:0.96

4/4 match. fast=True
